# Merge TCAV Outputs

This notebook loads all per-system TCAV CSV files from one output run, concatenates them into one long dataframe, and saves the merged CSV for downstream analysis.


In [ ]:
from pathlib import Path

import pandas as pd


In [ ]:
# ===== Config =====
PROJECT_ROOT = Path('/home/SpeakerRec/BioVoice')
TCAV_ROOT = PROJECT_ROOT / 'redimnet' / 'tcav' / 'captum_tcav' / 'asvspoof5'
OUTPUT_SUBDIR = 'subset_20spk_20utts_per_spk_one_logistic_head'
OUTPUT_DIR = TCAV_ROOT / 'output' / OUTPUT_SUBDIR

# Set to True if you want to drop A12 immediately.
EXCLUDE_A12 = False

MERGED_CSV_NAME = 'merged_tcav_long.csv'

print('OUTPUT_DIR =', OUTPUT_DIR)
print('Exists =', OUTPUT_DIR.exists())


In [ ]:
csv_paths = sorted(
    p for p in OUTPUT_DIR.glob('tcav_*.csv')
    if p.is_file() and not p.name.endswith('_wide.csv')
)

print('Found CSV files:', len(csv_paths))
for p in csv_paths[:10]:
    print(' ', p.name)
if len(csv_paths) > 10:
    print(' ...')


In [ ]:
required_cols = {
    'utt_id',
    'speaker_id',
    'split',
    'source_partition',
    'system_id',
    'target_class',
    'concept_name',
    'sign_count',
    'magnitude',
}

frames = []
for path in csv_paths:
    df = pd.read_csv(path)
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f'{path.name} is missing columns: {sorted(missing)}')
    df['source_file'] = path.name
    frames.append(df)

merged_df = pd.concat(frames, axis=0, ignore_index=True)

if EXCLUDE_A12:
    merged_df = merged_df[merged_df['system_id'] != 'A12'].copy()

merged_df = merged_df.sort_values(
    ['source_partition', 'system_id', 'target_class', 'speaker_id', 'utt_id', 'concept_name']
).reset_index(drop=True)

print('Merged rows =', len(merged_df))
print('Merged utterances =', merged_df['utt_id'].nunique())
print('Systems =', sorted(merged_df['system_id'].unique().tolist()))
print('Target classes =', sorted(merged_df['target_class'].unique().tolist()))


In [ ]:
summary_df = pd.DataFrame({
    'rows': [len(merged_df)],
    'utterances': [merged_df['utt_id'].nunique()],
    'speakers': [merged_df['speaker_id'].nunique()],
    'systems': [merged_df['system_id'].nunique()],
    'concepts': [merged_df['concept_name'].nunique()],
})
display(summary_df)

display(
    merged_df.groupby(['system_id', 'target_class'])['utt_id']
    .nunique()
    .rename('n_utterances')
    .reset_index()
    .sort_values(['system_id', 'target_class'])
)


In [ ]:
out_path = OUTPUT_DIR / MERGED_CSV_NAME
merged_df.to_csv(out_path, index=False)
print('Saved merged CSV to:', out_path)
